In [ ]:
from functools import partial
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

SEED = 42
# Información mutua con semilla fija (el estimador usa vecinos más cercanos con ruido aleatorio)
mi_score = partial(mutual_info_regression, random_state=SEED)

def mascara(index, ini, fin):
    return (index >= ini) & (index <= fin)

In [ ]:
f1 = FOLDS.iloc[0]
tr1 = mascara(X.index, f1.train_ini, f1.train_fin)

mi = pd.Series(mi_score(X[tr1], y[tr1]), index=X.columns, name="información mutua")
spearman = X[tr1].corrwith(y[tr1], method="spearman").rename("Spearman")
ranking = pd.concat([mi, spearman], axis=1).sort_values("información mutua", ascending=False)
ranking["posición"] = range(1, len(ranking) + 1)
display(ranking.round(3))

fig, ax = plt.subplots(figsize=(8, 9))
ranking["información mutua"].iloc[::-1].plot.barh(ax=ax)
ax.set_title(f"Información mutua con Parkinson del día τ\n(train {f1.train_ini.date()} → {f1.train_fin.date()})")
ax.set_xlabel("Información mutua (nats)")
plt.tight_layout(); plt.savefig(FIG_DIR / "03b_informacion_mutua.png", dpi=150); plt.show()

In [ ]:
grupo = (ranking.index.str.replace(r"\d+$", "", regex=True)
                      .str.replace("_lag", "_lag(k)").str.replace(r"_$", "", regex=True))
resumen_grupo = (ranking.assign(grupo=grupo)
                        .groupby("grupo")["información mutua"].agg(["mean", "max", "count"])
                        .sort_values("mean", ascending=False))
display(resumen_grupo.round(3))

In [ ]:
K_GRID = [3, 5, 8, 10, 15, 20, 25, 30, X.shape[1]]
folds_rep1 = FOLDS[FOLDS["rep"] == 1]

filas = []
for k in K_GRID:
    for f in folds_rep1.itertuples():
        tr = mascara(X.index, f.train_ini, f.train_fin)
        va = mascara(X.index, f.val_ini, f.val_fin)
        pipe_k = Pipeline([
            ("seleccion", SelectKBest(score_func=mi_score, k=k)),
            ("modelo", RandomForestRegressor(n_estimators=200, min_samples_leaf=20,
                                             random_state=SEED, n_jobs=-1)),
        ])
        pipe_k.fit(X[tr], y[tr])
        h = np.clip(pipe_k.predict(X[va]), 1e-6, None)
        filas.append({"k": k, "fold": f.fold,
                      "mse": np.mean((y[va] - h)**2),
                      "qlike": np.mean(y[va] / h + np.log(h))})

curva_k = pd.DataFrame(filas).groupby("k")[["mse", "qlike"]].agg(["mean", "std"])
display(curva_k.round(4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, m in zip(ax, ["mse", "qlike"]):
    a.errorbar(curva_k.index, curva_k[(m, "mean")], yerr=curva_k[(m, "std")], fmt="o-", capsize=3)
    a.set_xlabel("k (número de variables seleccionadas)"); a.set_title(f"{m.upper()} promedio en validación (rep. 1)")
plt.tight_layout(); plt.savefig(FIG_DIR / "03b_error_vs_k.png", dpi=150); plt.show()

In [ ]:
K_REF = 10   # valor de referencia para analizar estabilidad; el k definitivo lo elige Optuna

seleccion = {}
for f in FOLDS.itertuples():
    tr = mascara(X.index, f.train_ini, f.train_fin)
    sel = SelectKBest(score_func=mi_score, k=K_REF).fit(X[tr], y[tr])
    seleccion[f"R{f.rep}·F{f.fold}"] = sel.get_support()

matriz_sel = pd.DataFrame(seleccion, index=X.columns)
frecuencia = matriz_sel.sum(axis=1).sort_values(ascending=False)
display(frecuencia[frecuencia > 0].to_frame(f"veces seleccionada (de {len(FOLDS)} folds)"))

usadas = frecuencia[frecuencia > 0].index
fig, ax = plt.subplots(figsize=(10, 0.35 * len(usadas) + 1.5))
ax.imshow(matriz_sel.loc[usadas].astype(int), cmap="Blues", aspect="auto")
ax.set_yticks(range(len(usadas)), usadas, fontsize=8)
ax.set_xticks(range(matriz_sel.shape[1]), matriz_sel.columns, rotation=45, fontsize=8)
ax.set_title(f"Variables seleccionadas en cada fold (k = {K_REF})")
plt.tight_layout(); plt.savefig(FIG_DIR / "03b_seleccion_por_fold.png", dpi=150); plt.show()

In [ ]:
def crear_pipeline(k=10, **params_rf):
    # Pipeline: selección tipo filter (información mutua) + Random Forest
    return Pipeline([
        ("seleccion", SelectKBest(score_func=mi_score, k=k)),
        ("modelo", RandomForestRegressor(random_state=SEED, n_jobs=-1, **params_rf)),
    ])

pipe = crear_pipeline(k=K_REF)
pipe

In [ ]:
ranking.to_csv(PROC / "fs_ranking_fold1.csv")
curva_k.to_csv(PROC / "fs_error_vs_k.csv")
matriz_sel.to_csv(PROC / "fs_seleccion_por_fold.csv")
print("Guardado en", PROC)